In [1]:
import pandas as pd
import numpy as np

# 1. Đọc file CSV (Thay tên file của ông vào đây)
# Giả sử file này xuất từ PlotJuggler chứa cả tọa độ (odom) và vận tốc (cmd_vel)
df = pd.read_csv('theta.csv') 
df.columns = df.columns.str.strip()

# 2. Tự động tìm cột (Kiểm tra lại tên cột trong file CSV của ông nhé)
time_raw = df.iloc[:, 0].to_numpy()
time = time_raw - time_raw[0]

vx = df[[c for c in df.columns if 'linear/x' in c][0]].to_numpy()
vy = df[[c for c in df.columns if 'linear/y' in c][0]].to_numpy()
wz = df[[c for c in df.columns if 'angular/z' in c][0]].to_numpy()

# Nếu có cột x, y từ Odom để tính Path Length
try:
    x = df[[c for c in df.columns if 'position/x' in c][0]].to_numpy()
    y = df[[c for c in df.columns if 'position/y' in c][0]].to_numpy()
    path_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))
except IndexError:
    path_length = "Không có data Odom trong file CSV"

# 3. TÍNH TOÁN CÁC METRICS HÀN LÂM
# A. Travel Time
travel_time = time[-1]

# B. Vận tốc tịnh tiến tổng hợp (Linear Velocity)
v_linear = np.sqrt(vx**2 + vy**2)

# C. Smoothness (Độ lệch chuẩn của Vận tốc)
std_wz = np.std(wz)
std_v = np.std(v_linear)

# D. TÍNH JERK (ĐỘ GIẬT) = Đạo hàm bậc 2 của Vận tốc
# Để tránh nhiễu do chia cho dt quá nhỏ, ta dùng dt trung bình
dt = np.mean(np.diff(time))

# Tính Gia tốc (Acceleration)
accel_v = np.diff(v_linear) / dt
accel_wz = np.diff(wz) / dt

# Tính Jerk (Đạo hàm của gia tốc)
jerk_v = np.diff(accel_v) / dt
jerk_wz = np.diff(accel_wz) / dt

# Lấy giá trị RMS (Root Mean Square) hoặc Std của Jerk làm đại diện
jerk_v_std = np.std(jerk_v)
jerk_wz_std = np.std(jerk_wz)

# 4. IN KẾT QUẢ ĐỂ ĐIỀN VÀO BẢNG
print("="*40)
print(f"BÁO CÁO METRICS CHO BÀI BÁO ASME")
print("="*40)
print(f"1. Path Length (m):        {path_length}")
print(f"2. Travel Time (s):        {travel_time:.2f}")
print(f"3. Std Dev (wz) (rad/s):   {std_wz:.4f}  <-- Độ ổn định góc")
print(f"4. Std Dev (V) (m/s):      {std_v:.4f}  <-- Độ ổn định tịnh tiến")
print(f"5. Linear Jerk (m/s^3):    {jerk_v_std:.4f}  <-- Hao mòn động cơ kéo")
print(f"6. Angular Jerk (rad/s^3): {jerk_wz_std:.4f}  <-- Hao mòn động cơ lái")
print("="*40)

BÁO CÁO METRICS CHO BÀI BÁO ASME
1. Path Length (m):        nan
2. Travel Time (s):        79.06
3. Std Dev (wz) (rad/s):   nan  <-- Độ ổn định góc
4. Std Dev (V) (m/s):      nan  <-- Độ ổn định tịnh tiến
5. Linear Jerk (m/s^3):    nan  <-- Hao mòn động cơ kéo
6. Angular Jerk (rad/s^3): nan  <-- Hao mòn động cơ lái


In [5]:
import pandas as pd
import numpy as np

# 1. Đọc file CSV
df = pd.read_csv('theta.csv') 
df.columns = df.columns.str.strip()

# =========================================================
# PHẦN 1: TÍNH THÔNG SỐ ĐIỀU KHIỂN TỪ /CMD_VEL_NAV
# =========================================================
# Lấy tên cột tự động
col_time = df.columns[0]
col_vx = [c for c in df.columns if 'linear/x' in c][0]
col_vy = [c for c in df.columns if 'linear/y' in c][0]
col_wz = [c for c in df.columns if 'angular/z' in c][0]

# LỌC SẠCH NaN CHỈ CHO NHÓM CMD_VEL
df_cmd = df[[col_time, col_vx, col_vy, col_wz]].dropna()

time_cmd = df_cmd[col_time].to_numpy()
vx = df_cmd[col_vx].to_numpy()
vy = df_cmd[col_vy].to_numpy()
wz = df_cmd[col_wz].to_numpy()

# Tính vận tốc tổng
v_linear = np.sqrt(vx**2 + vy**2)

# TÌM THỜI GIAN DI CHUYỂN THỰC TẾ (Bỏ qua lúc đứng yên)
moving_indices = np.where(v_linear > 0.01)[0]
start_idx = moving_indices[0]
end_idx = moving_indices[-1]

# Travel time chuẩn xác
start_time = time_cmd[start_idx]
end_time = time_cmd[end_idx]
travel_time = end_time - start_time

# Lọc chỉ lấy data trong lúc xe THỰC SỰ CHẠY để tính độ lệch chuẩn
time_m = time_cmd[start_idx:end_idx+1]
v_lin_m = v_linear[start_idx:end_idx+1]
wz_m = wz[start_idx:end_idx+1]

std_v = np.std(v_lin_m)
std_wz = np.std(wz_m)

# Tính Jerk (Độ giật)
dt = np.mean(np.diff(time_m))
accel_v = np.diff(v_lin_m) / dt
accel_wz = np.diff(wz_m) / dt

jerk_v_std = np.std(np.diff(accel_v) / dt)
jerk_wz_std = np.std(np.diff(accel_wz) / dt)

# =========================================================
# PHẦN 2: TÍNH QUÃNG ĐƯỜNG TỪ /ODOM
# =========================================================
col_x = [c for c in df.columns if 'position/x' in c][0]
col_y = [c for c in df.columns if 'position/y' in c][0]

# LỌC SẠCH NaN CHỈ CHO NHÓM ODOM
df_odom = df[[col_time, col_x, col_y]].dropna()

# Lọc Odom khớp đúng với khoảng thời gian xe chạy
df_odom_moving = df_odom[(df_odom[col_time] >= start_time) & (df_odom[col_time] <= end_time)]

x = df_odom_moving[col_x].to_numpy()
y = df_odom_moving[col_y].to_numpy()

# Tính quãng đường
path_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))

# =========================================================
# IN KẾT QUẢ ĐIỀN BẢNG
# =========================================================
print("="*45)
print(f"BÁO CÁO METRICS CHO THETA*")
print("="*45)
print(f"1. Path Length (m):        {path_length:.3f}")
print(f"2. Travel Time (s):        {travel_time:.2f}")
print(f"3. Std Dev (wz) (rad/s):   {std_wz:.4f}  <-- Độ ổn định góc")
print(f"4. Std Dev (V) (m/s):      {std_v:.4f}  <-- Độ ổn định tịnh tiến")
print(f"5. Linear Jerk (m/s^3):    {jerk_v_std:.4f}  <-- Hao mòn động cơ kéo")
print(f"6. Angular Jerk (rad/s^3): {jerk_wz_std:.4f}  <-- Hao mòn động cơ lái")
print("="*45)

BÁO CÁO METRICS CHO THETA*
1. Path Length (m):        8.649
2. Travel Time (s):        22.25
3. Std Dev (wz) (rad/s):   0.1509  <-- Độ ổn định góc
4. Std Dev (V) (m/s):      0.1450  <-- Độ ổn định tịnh tiến
5. Linear Jerk (m/s^3):    3.5672  <-- Hao mòn động cơ kéo
6. Angular Jerk (rad/s^3): 3.1613  <-- Hao mòn động cơ lái


In [6]:
import pandas as pd
import numpy as np

# 1. Đọc file CSV
df = pd.read_csv('dijkstra.csv') 
df.columns = df.columns.str.strip()

# =========================================================
# PHẦN 1: TÍNH THÔNG SỐ ĐIỀU KHIỂN TỪ /CMD_VEL_NAV
# =========================================================
# Lấy tên cột tự động
col_time = df.columns[0]
col_vx = [c for c in df.columns if 'linear/x' in c][0]
col_vy = [c for c in df.columns if 'linear/y' in c][0]
col_wz = [c for c in df.columns if 'angular/z' in c][0]

# LỌC SẠCH NaN CHỈ CHO NHÓM CMD_VEL
df_cmd = df[[col_time, col_vx, col_vy, col_wz]].dropna()

time_cmd = df_cmd[col_time].to_numpy()
vx = df_cmd[col_vx].to_numpy()
vy = df_cmd[col_vy].to_numpy()
wz = df_cmd[col_wz].to_numpy()

# Tính vận tốc tổng
v_linear = np.sqrt(vx**2 + vy**2)

# TÌM THỜI GIAN DI CHUYỂN THỰC TẾ (Bỏ qua lúc đứng yên)
moving_indices = np.where(v_linear > 0.01)[0]
start_idx = moving_indices[0]
end_idx = moving_indices[-1]

# Travel time chuẩn xác
start_time = time_cmd[start_idx]
end_time = time_cmd[end_idx]
travel_time = end_time - start_time

# Lọc chỉ lấy data trong lúc xe THỰC SỰ CHẠY để tính độ lệch chuẩn
time_m = time_cmd[start_idx:end_idx+1]
v_lin_m = v_linear[start_idx:end_idx+1]
wz_m = wz[start_idx:end_idx+1]

std_v = np.std(v_lin_m)
std_wz = np.std(wz_m)

# Tính Jerk (Độ giật)
dt = np.mean(np.diff(time_m))
accel_v = np.diff(v_lin_m) / dt
accel_wz = np.diff(wz_m) / dt

jerk_v_std = np.std(np.diff(accel_v) / dt)
jerk_wz_std = np.std(np.diff(accel_wz) / dt)

# =========================================================
# PHẦN 2: TÍNH QUÃNG ĐƯỜNG TỪ /ODOM
# =========================================================
col_x = [c for c in df.columns if 'position/x' in c][0]
col_y = [c for c in df.columns if 'position/y' in c][0]

# LỌC SẠCH NaN CHỈ CHO NHÓM ODOM
df_odom = df[[col_time, col_x, col_y]].dropna()

# Lọc Odom khớp đúng với khoảng thời gian xe chạy
df_odom_moving = df_odom[(df_odom[col_time] >= start_time) & (df_odom[col_time] <= end_time)]

x = df_odom_moving[col_x].to_numpy()
y = df_odom_moving[col_y].to_numpy()

# Tính quãng đường
path_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))

# =========================================================
# IN KẾT QUẢ ĐIỀN BẢNG
# =========================================================
print("="*45)
print(f"BÁO CÁO METRICS CHO DIJKSTRA")
print("="*45)
print(f"1. Path Length (m):        {path_length:.3f}")
print(f"2. Travel Time (s):        {travel_time:.2f}")
print(f"3. Std Dev (wz) (rad/s):   {std_wz:.4f}  <-- Độ ổn định góc")
print(f"4. Std Dev (V) (m/s):      {std_v:.4f}  <-- Độ ổn định tịnh tiến")
print(f"5. Linear Jerk (m/s^3):    {jerk_v_std:.4f}  <-- Hao mòn động cơ kéo")
print(f"6. Angular Jerk (rad/s^3): {jerk_wz_std:.4f}  <-- Hao mòn động cơ lái")
print("="*45)

BÁO CÁO METRICS CHO DIJKSTRA
1. Path Length (m):        9.451
2. Travel Time (s):        24.15
3. Std Dev (wz) (rad/s):   0.1467  <-- Độ ổn định góc
4. Std Dev (V) (m/s):      0.1383  <-- Độ ổn định tịnh tiến
5. Linear Jerk (m/s^3):    2.7673  <-- Hao mòn động cơ kéo
6. Angular Jerk (rad/s^3): 2.9290  <-- Hao mòn động cơ lái


In [7]:
import pandas as pd
import numpy as np

# 1. Đọc file CSV
df = pd.read_csv('astar.csv') 
df.columns = df.columns.str.strip()

# =========================================================
# PHẦN 1: TÍNH THÔNG SỐ ĐIỀU KHIỂN TỪ /CMD_VEL_NAV
# =========================================================
# Lấy tên cột tự động
col_time = df.columns[0]
col_vx = [c for c in df.columns if 'linear/x' in c][0]
col_vy = [c for c in df.columns if 'linear/y' in c][0]
col_wz = [c for c in df.columns if 'angular/z' in c][0]

# LỌC SẠCH NaN CHỈ CHO NHÓM CMD_VEL
df_cmd = df[[col_time, col_vx, col_vy, col_wz]].dropna()

time_cmd = df_cmd[col_time].to_numpy()
vx = df_cmd[col_vx].to_numpy()
vy = df_cmd[col_vy].to_numpy()
wz = df_cmd[col_wz].to_numpy()

# Tính vận tốc tổng
v_linear = np.sqrt(vx**2 + vy**2)

# TÌM THỜI GIAN DI CHUYỂN THỰC TẾ (Bỏ qua lúc đứng yên)
moving_indices = np.where(v_linear > 0.01)[0]
start_idx = moving_indices[0]
end_idx = moving_indices[-1]

# Travel time chuẩn xác
start_time = time_cmd[start_idx]
end_time = time_cmd[end_idx]
travel_time = end_time - start_time

# Lọc chỉ lấy data trong lúc xe THỰC SỰ CHẠY để tính độ lệch chuẩn
time_m = time_cmd[start_idx:end_idx+1]
v_lin_m = v_linear[start_idx:end_idx+1]
wz_m = wz[start_idx:end_idx+1]

std_v = np.std(v_lin_m)
std_wz = np.std(wz_m)

# Tính Jerk (Độ giật)
dt = np.mean(np.diff(time_m))
accel_v = np.diff(v_lin_m) / dt
accel_wz = np.diff(wz_m) / dt

jerk_v_std = np.std(np.diff(accel_v) / dt)
jerk_wz_std = np.std(np.diff(accel_wz) / dt)

# =========================================================
# PHẦN 2: TÍNH QUÃNG ĐƯỜNG TỪ /ODOM
# =========================================================
col_x = [c for c in df.columns if 'position/x' in c][0]
col_y = [c for c in df.columns if 'position/y' in c][0]

# LỌC SẠCH NaN CHỈ CHO NHÓM ODOM
df_odom = df[[col_time, col_x, col_y]].dropna()

# Lọc Odom khớp đúng với khoảng thời gian xe chạy
df_odom_moving = df_odom[(df_odom[col_time] >= start_time) & (df_odom[col_time] <= end_time)]

x = df_odom_moving[col_x].to_numpy()
y = df_odom_moving[col_y].to_numpy()

# Tính quãng đường
path_length = np.sum(np.sqrt(np.diff(x)**2 + np.diff(y)**2))

# =========================================================
# IN KẾT QUẢ ĐIỀN BẢNG
# =========================================================
print("="*45)
print(f"BÁO CÁO METRICS CHO A*")
print("="*45)
print(f"1. Path Length (m):        {path_length:.3f}")
print(f"2. Travel Time (s):        {travel_time:.2f}")
print(f"3. Std Dev (wz) (rad/s):   {std_wz:.4f}  <-- Độ ổn định góc")
print(f"4. Std Dev (V) (m/s):      {std_v:.4f}  <-- Độ ổn định tịnh tiến")
print(f"5. Linear Jerk (m/s^3):    {jerk_v_std:.4f}  <-- Hao mòn động cơ kéo")
print(f"6. Angular Jerk (rad/s^3): {jerk_wz_std:.4f}  <-- Hao mòn động cơ lái")
print("="*45)

BÁO CÁO METRICS CHO A*
1. Path Length (m):        8.834
2. Travel Time (s):        23.35
3. Std Dev (wz) (rad/s):   0.1497  <-- Độ ổn định góc
4. Std Dev (V) (m/s):      0.1412  <-- Độ ổn định tịnh tiến
5. Linear Jerk (m/s^3):    3.3943  <-- Hao mòn động cơ kéo
6. Angular Jerk (rad/s^3): 3.1783  <-- Hao mòn động cơ lái


In [ ]:
# =========================================================
# VẼ HÌNH ĐÃ ĐƯỢC LÀM MƯỢT VÀ CHUẨN KÍCH THƯỚC BÀI BÁO
# =========================================================
plt.style.use('seaborn-whitegrid')
plt.rcParams['font.family'] = 'serif'

# 1. SET SIZE CHỮ CƠ BẢN TO LÊN
plt.rcParams['font.size'] = 14
plt.rcParams['axes.linewidth'] = 1.5

# 2. CHỈNH FIGSIZE VỪA VỚI 1 CỘT (Ngang 6 inch, Dọc 4.5 inch)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 4.5), sharex=True)

# 3. TĂNG ĐỘ DÀY NÉT VẼ (linewidth = 2.5)
ax1.plot(time, vx, color='#d62728', linewidth=2.5, label=r'$v_x$ (Forward)')
ax1.plot(time, vy, color='#1f77b4', linewidth=2.5, label=r'$v_y$ (Strafe)')

ax1.set_ylabel('Linear Vel (m/s)', fontweight='bold', fontsize=12)

# Chỉnh cho Legend nhỏ gọn lại một chút kẻo che mất đường biểu diễn
ax1.legend(loc='upper right', frameon=True, edgecolor='black', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_title(r'Theta*', fontweight='bold', fontsize=14) # Nhớ đổi tên chỗ này cho đúng đồ thị

# ĐỒ THỊ 2: VẬN TỐC GÓC
ax2.plot(time, wz, color='#2ca02c', linewidth=2.5, label=r'$\omega_z$ (Yaw Rate)')
ax2.set_xlabel('Time (s)', fontweight='bold', fontsize=12)
ax2.set_ylabel('Angular Vel (rad/s)', fontweight='bold', fontsize=12)
ax2.legend(loc='upper right', frameon=True, edgecolor='black', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()

# 4. LƯU THÀNH ĐUÔI .PDF ĐỂ ẢNH ĐẠT ĐỘ NÉT TUYỆT ĐỐI KHÔNG BAO GIỜ MỜ
plt.savefig('theta2.pdf', format='pdf', bbox_inches='tight') 
# Trong file LaTeX ông nhúng lệnh \includegraphics{theta2.pdf} bình thường nhé